# Quick Start: Build and Train an RG-Net

End-to-end demo: critical init → train → evaluate. Takes ~2 min on CPU.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Build RG-Net with Critical Initialization

In [ ]:
from src.architectures.rg_net.rg_net import RGNetStandard, RGLayer
from src.datasets.hierarchical_dataset import HierarchicalGaussianDataset, DatasetConfig
from src.training.trainer import Trainer, TrainingConfig
from src.core.correlation.two_point import chi1_gauss_hermite
import torch

# Verify critical init
sigma_w, sigma_b = 1.4, 0.3
chi1 = chi1_gauss_hermite(sigma_w**2, "tanh")
print(f"Critical init: sigma_w={sigma_w}, sigma_b={sigma_b}")
print(f"chi1 = {chi1:.4f} (should be near 1.0 for deep signal propagation)")

# Build model
model = RGNetStandard(
    input_dim=64, hidden_dim=64, output_dim=4,
    depth=10, activation="tanh"
)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: RGNetStandard, depth=10, width=64, params={n_params:,}")


## Fast-Track Training (2 epochs)

In [ ]:
from torch.utils.data import DataLoader

cfg_data = DatasetConfig(n_samples=1000, input_dim=64, n_classes=4, xi_data=10.0, fast_track=True)
ds = HierarchicalGaussianDataset(cfg_data)
n_train = int(0.8 * len(ds))
train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, len(ds)-n_train])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)

cfg_train = TrainingConfig(n_epochs=2, lr=1e-3, fast_track=True)
trainer = Trainer(model, cfg_train, device=device)
result = trainer.train(train_loader, val_loader)
print(f"\nTraining complete: best_val_acc={result.best_val_acc:.4f}, time={result.elapsed_s:.1f}s")
